<a href="https://colab.research.google.com/github/voiceform-lt/.github/blob/main/segment_single_label_recording.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

input_wav = next(iter(uploaded.keys()))

print("Naudojamas failas:", input_wav)

In [ ]:
import os
import numpy as np
import librosa
import soundfile as sf

OUTPUT_DIR = "zodziai_sukalibruoti"
TARGET_SR = 16000
MIN_WORD_DURATION_SEC = 0.15
SILENCE_DB = 20  # Paliekame optimalų jautrumą

# GRIEŽTAS MOKSLINIS REIKALAVIMAS: failo ilgis turi idealiai atitikti modelio STFT
TARGET_SAMPLES = 16144  # 1.009 sekundės

# --- NAUJAS PARAMETRAS DAUGIASKIEMENIŲ ŽODŽIŲ APSAUGAI ---
# Jei tarp dviejų garsų yra mažesnis nei šis tarpas (0.35s), laikome, kad tai tas pats žodis.
MAX_GAP_BETWEEN_SYLLABLES_SEC = 0.35
max_gap_samples = int(MAX_GAP_BETWEEN_SYLLABLES_SEC * TARGET_SR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Užkrauname audio srautą
audio, sr = librosa.load(input_wav, sr=TARGET_SR, mono=True)

# Pirmasis grubus skaidymas naudojant didesnį langą
intervals = librosa.effects.split(
    audio,
    top_db=SILENCE_DB,
    frame_length=4096,
    hop_length=512
)

# --- INTELEKTUALUS SEGMENTŲ SUJUNGIMO ALGORITMAS ---
valid_intervals = []
if len(intervals) > 0:
    current_start, current_end = intervals[0]

    for next_start, next_end in intervals[1:]:
        # Skaičiuojame pauzės ilgį tarp dabartinio segmento pabaigos ir kito pradžios
        gap = next_start - current_end

        if gap < max_gap_samples:
            # Pauzė per trumpa (skiemuo žodžio viduje) -> Pailginame dabartinį segmentą
            current_end = next_end
            print(f"[DSP INFO] Sujungiami skiemenys. Pauzė buvo: {gap/TARGET_SR:.3f} s.")
        else:
            # Pauzė pakankamai didelė -> Užfiksuojame pilną žodį ir pradedame naują
            valid_intervals.append((current_start, current_end))
            current_start, current_end = next_start, next_end

    # Įdedame paskutinį segmentą
    valid_intervals.append((current_start, current_end))
else:
    valid_intervals = intervals

# --- FAILŲ GENERAVIMAS IR APDOROJIMAS ---
base_name = os.path.splitext(os.path.basename(input_wav))[0]
min_samples = int(MIN_WORD_DURATION_SEC * TARGET_SR)
count = 0

for start, end in valid_intervals:
    segment = audio[start:end]

    if len(segment) < min_samples:
        continue

    if len(segment) > TARGET_SAMPLES:
        print(f"[ĮSPĖJIMAS] Žodis {count+1} yra per ilgas ({len(segment)} samples). Nukerpama galūnė.")
        final_segment = segment[:TARGET_SAMPLES]
    else:
        # Augmentacija su atsitiktiniu laiko poslinkiu
        total_pad = TARGET_SAMPLES - len(segment)
        np.random.seed(count)
        pad_left = np.random.randint(0, total_pad + 1)
        pad_right = total_pad - pad_left
        final_segment = np.pad(segment, (pad_left, pad_right), mode='constant')

    count += 1
    out_name = f"{base_name}_{count:04d}.wav"
    out_path = os.path.join(OUTPUT_DIR, out_name)

    sf.write(out_path, final_segment, TARGET_SR, subtype="PCM_16")

print(f"\n[SĖKMĖ] Apdorojimas baigtas! Sukurti {count} vientisi failai.")

In [ ]:
import shutil

zip_name = f"{base_name}_zodziai"

shutil.make_archive(zip_name, "zip", OUTPUT_DIR)

files.download(f"{zip_name}.zip")